# Cinemática Inversa Numérica
---

In [137]:
import numpy as np
from numpy import float32, float64, int32, int64

from dataclasses import dataclass
from typing import Self

Utilizamos alguna semilla fila, para que las pruebas a pesar de ser aleatoria, sean determinista para cada corrida del código y utilizar los resultados estadísticos

In [138]:
np.random.seed(0)

## Definición del Robot
---
En esta sección vamos a definir la Pose, el Robot IRB140 con su configuración y dimensiones, para usarlos a lo largo del trabajo. Se define la cinemática directa del mismo ya que será necesario para calcula la cinemática inversa de forma numérica

In [139]:
@dataclass
class Pose:
  rotacion: np.ndarray
  translacion: np.ndarray

  @classmethod
  def identidad(cls) -> Self:
    return Pose(
      np.eye(3),
      np.zeros(3),
    )

  def __mul__(self, pose: Self) -> Self:
    return Pose(
      self.rotacion @ pose.rotacion,
      self.rotacion @ pose.translacion + self.translacion,
    )

  def distancia_cuadrado(self, pose: Self) -> float64:
    diff = self.translacion - pose.translacion
    return diff[0]**2 + diff[1]**2 + diff[2]**2

In [140]:
def DH(theta: float64, d: float64, a: float64, alpha: float64) -> Pose:
    cth = np.cos(theta)
    sth = np.sin(theta)
    ca = np.cos(alpha)
    sa = np.sin(alpha)

    return Pose(np.array([
        [cth, -sth*ca,  sth*sa],
        [sth,  cth*ca, -cth*sa],
        [  0,     sa,       ca],
    ]), np.array([ a*cth, a*sth, d ]))

In [141]:
@dataclass
class Dimensiones:
  d1: float64
  a1: float64
  a2: float64
  d4: float64
  d6: float64

@dataclass
class Configuracion:
  c1: bool # True es lo que vimos como 1, y Flase como -1
  c2: bool
  c3: bool

class IRB140:
  def __init__(self, dim: Dimensiones, ABase0: Pose | None = None, A6Tool: Pose | None = None):
    self.dim = dim

    self.A01 = lambda q1: DH(q1, dim.d1, dim.a1, -np.pi/2)
    self.A12 = lambda q2: DH(q2,      0, dim.a2,        0)
    self.A23 = lambda q3: DH(q3,      0,      0,  np.pi/2)
    self.A34 = lambda q4: DH(q4, dim.d4,      0, -np.pi/2)
    self.A45 = lambda q5: DH(q5,      0,      0,  np.pi/2)
    self.A56 = lambda q6: DH(q6, dim.d6,      0,        0)

    self.ABase0 = Pose.identidad() if ABase0 is None else ABase0
    self.A6Tool = Pose.identidad() if A6Tool is None else A6Tool

  def cinematica_directa(self, q: np.ndarray) -> Pose:
    q1, q2, q3, q4, q5, q6 = tuple(q.tolist())
    A06 = self.A01(q1) * self.A12(q2) * self.A23(q3) * self.A34(q4) * self.A45(q5) * self.A56(q6)
    return self.ABase0 * A06 * self.A6Tool

  def configuracion(self, q: np.ndarray, epsilon: float64 = 1e-6) -> Configuracion | None:
    _, q2, q3, _, q5, _ = tuple(q.tolist())

    config1 = self.dim.d4 * np.sin(q2 + q3) + self.dim.a2 * np.cos(q2) + self.dim.a1
    config2 = np.cos(q3)
    config3 = q5 + 0.0001

    if abs(config1) < epsilon or abs(config2) < epsilon or abs(config3) < epsilon:
      return None

    return Configuracion(bool(config1 > 0), bool(config2 > 0), bool(config3 > 0))

irb140 = IRB140(Dimensiones(0, 70, 360, 380, 0))

## Cinemática inversa naïve
---
La forma que plantearemos para resolver la cinemática inversa es a partir de generar una función de costo $J(\bar{q})$, la cual el algoritmo va a intentar minimizar

Por lo tanto, para resolver la minimización utilizaremos el algoritmo del gradiente descendiente, el cual es un método iterativo que dado una semilla $\bar{q}_0$, la va actualizando en dirección opuesta al gradiente $\nabla J(\bar{q})$ de la siguiente manera $$ \bar{q}_{n + 1} = \bar{q}_{n} - \alpha \nabla J(\bar{q}) $$ donde $\alpha$ es un hiperparámetro del algoritmo

Finalmente, para lograr que el mínimo que encuentre el algoritmo sea la Pose destino $\text{Pose}_{dest}$ que estamos buscando, plantearemos la función de costo como $$ J(\bar{q}) = \left\lVert \text{Pose}_{base}^{tool}(\bar{q}) - \text{Pose}_{dest} \right\rVert^2 $$ donde $\text{Pose}_{base}^{tool}(\bar{q})$ representa la cinemática directa   

In [142]:
def funcion_costo(pose_origen: Pose, pose_destino: Pose) -> np.float64:
  return pose_origen.distancia_cuadrado(pose_destino)

In [143]:
def cinematima_inversa(
  pose_destino: Pose,
  alfa: float64,
  epsilon: float64 = 1e-2,
  max_iteraciones: int32 = 1000,
  variacion: float64 = 1e-4,
) -> tuple[np.ndarray, bool]:
  # Semilla inicial al algoritmo
  q = np.array([0., 0., 0., 0., 0., 0.])

  termina_correctamente = False
  for _ in range(max_iteraciones):
    costo = funcion_costo(irb140.cinematica_directa(q), pose_destino)
    if costo < epsilon:
      termina_correctamente = True
      break

    gradiente = np.zeros(6)
    for i in range(6):
      delta_q = np.zeros(6)
      delta_q[i] += variacion

      delta_costo = funcion_costo(irb140.cinematica_directa(q + delta_q), pose_destino) - costo

      gradiente[i] = delta_costo / variacion

    q -= alfa * gradiente

  return np.mod(q + np.pi, 2 * np.pi) - np.pi, termina_correctamente

Para confirmar el funcionamiento, realizaremos pruebas generando $100$ valores de $\bar{q}$ aleatorios, los cuales nos aseguramos que tenga su correspondiente inversa asegurandonos que tenga una configuración valida, y por cinemática directa calcular su correspondiente pose, la cual será la pose destino utilizada en el calculo de la cinemática inversa

In [144]:
def generar_angulos() -> tuple[np.ndarray, Configuracion]:
  q = np.zeros(6)
  config = None

  while config is None:
    # definimos los q ∈ [-π, π]
    q = 2 * np.pi * (np.random.rand(6) - 0.5)
    q[3:] *= 0
    config = irb140.configuracion(q)

  return q, config


In [145]:
CANT_PRUEBAS = 100
print(f"Numero de pruebas: {CANT_PRUEBAS}")

cant_acertada = 0
for _ in range(CANT_PRUEBAS):
  q_destino, _ = generar_angulos()
  q_obtenido, _ = cinematima_inversa(irb140.cinematica_directa(q_destino), 1e-6, max_iteraciones = 100)

  if np.linalg.norm(q_destino - q_obtenido) < 1e-1:
    cant_acertada += 1

print(f"\tResultado: {cant_acertada}/{CANT_PRUEBAS}")

Numero de pruebas: 100
	Resultado: 10/100


Se obtiene el resultado de $10$ entre las $100$ pruebas que se intentaron, esto muestra que a pesar de obtener resultados correctos, el algoritmo necesita más información para obtener el $100\%$ de aciertos

Esto ocurre ya que el problema que resuelve el gradiente descendiente es de minimización. Como conocemos el robot, sabemos que para una dada pose pueden existir multiples configuraciones del robot que lleven el end-effector a la pose destino, y esta implementación de la cinemática inversa no puede distinguir entre estas configuraciones dandonos como resultado los $\bar{q}$ más cercanos a la configuración de la semilla inicial

## Cinemática inversa con más información
---
Como vimos se tiene que incorporar más información al algoritmo para obtener el $100\%$ de aciertos. Esta información puede ser dada por la configuración de la pose destino, por lo que probarémos utilizar una semilla que tenga la misma configuración

In [146]:
def semilla(config: Configuracion) -> np.ndarray:
  match (config.c1, config.c2, config.c3):
    case (( True,  True,  True)): return np.array([ 0., 0., 0., 0., 0., 0., ])
    case (( True,  True, False)): return np.array([ 0., 0., 0., 0., 0., 0., ])
    case (( True, False,  True)): return np.array([ 0., 0., 0., 0., 0., 0., ])
    case (( True, False, False)): return np.array([ 0., 0., 0., 0., 0., 0., ])
    case ((False,  True,  True)): return np.array([ 0., 0., 0., 0., 0., 0., ])
    case ((False,  True, False)): return np.array([ 0., 0., 0., 0., 0., 0., ])
    case ((False, False,  True)): return np.array([ 0., 0., 0., 0., 0., 0., ])
    case ((False, False, False)): return np.array([ 0., 0., 0., 0., 0., 0., ])

def cinematima_inversa(
  pose_destino: Pose,
  config: Configuracion,
  alfa: float64,
  epsilon: float64 = 1e-2,
  max_iteraciones: int32 = 1000,
  variacion: float64 = 1e-4,
) -> tuple[np.ndarray, bool]:
  # Semilla inicial al algoritmo determinada por la configuración
  q = semilla(config)

  termina_correctamente = False
  for _ in range(max_iteraciones):
    costo = funcion_costo(irb140.cinematica_directa(q), pose_destino)
    if costo < epsilon:
      termina_correctamente = True
      break

    gradiente = np.zeros(6)
    for i in range(6):
      delta_q = np.zeros(6)
      delta_q[i] += variacion

      delta_costo = funcion_costo(irb140.cinematica_directa(q + delta_q), pose_destino) - costo

      gradiente[i] = delta_costo / variacion

    q -= alfa * gradiente

  return np.mod(q + np.pi, 2 * np.pi) - np.pi, termina_correctamente

In [147]:
CANT_PRUEBAS = 100
print(f"Numero de pruebas con configuración: {CANT_PRUEBAS}")

cant_acertada = 0
for _ in range(CANT_PRUEBAS):
  q_destino, config = generar_angulos()
  q_obtenido, ok = cinematima_inversa(irb140.cinematica_directa(q_destino), config, 1e-6, max_iteraciones = 100)

  if np.linalg.norm(q_destino - q_obtenido) < 1e-1:
    cant_acertada += 1

print(f"\tResultado: {cant_acertada}/{CANT_PRUEBAS}")

Numero de pruebas con configuración: 100
	Resultado: 7/100
